# Building a Spam Classifier with Naive Bayes

In [46]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [47]:
X_train = np.loadtxt('train-features.txt')
y_train = np.loadtxt('train-labels.txt')

X_val = np.loadtxt('val-features.txt')
y_val = np.loadtxt('val-labels.txt')

print("Training features:", X_train.shape)
print("Training labels:", y_train.shape)
print("Validation features:", X_val.shape)
print("Validation labels:", y_val.shape)

Training features: (700, 2500)
Training labels: (700,)
Validation features: (260, 2500)
Validation labels: (260,)


### PART a)

In [48]:
n_spam = np.sum(y_train==1)
n_nonspam = np.sum(y_train==0)

n_tot = len(y_train)

spam_percentage = (n_spam/n_tot)*100
nonspam_percentage = (n_nonspam/n_tot)*100

print("Spam emails:", n_spam)
print("Non-spam emails:", n_nonspam)
print("Spam percentage:", spam_percentage)
print("Non-spam percentage:", nonspam_percentage)

Spam emails: 350
Non-spam emails: 350
Spam percentage: 50.0
Non-spam percentage: 50.0


**Part (a) — Class Distribution**

The training dataset contains 700 emails, of which **350** are spam and **350** are non-spam.

The percentage of spam emails is:

Spam percentage = (Number of spam emails / 700) × 100 = **50%**.

Therefore, the training dataset is **balanced**

### Part (b)

In [1]:
n_classes = len(np.unique(y_train))
vocab_size = X_train.shape[1]

n_free_parameters = (n_classes - 1) + n_classes * (vocab_size - 1)

print("Number of classes:", n_classes)
print("Vocabulary size:", vocab_size)
print("Independent class prior parameters:", n_classes - 1)
print("Independent word probability parameters:",
      n_classes * (vocab_size - 1))
print("Total independent parameters:", n_free_parameters)

NameError: name 'np' is not defined

**Part (b) — Number of Free Parameters**

The Multinomial Naive Bayes model has two classes and a vocabulary of 2500 words.

The class prior probabilities contribute \(2-1=1\) free parameter because their sum must equal one.

For each class, the 2500 conditional word probabilities must also sum to one. Therefore, each class contributes \(2500-1=2499\) free parameters.

The total number of independent parameters is:

$$
N_{\text{free}}=(2-1)+2(2500-1)=\boxed{4999}
$$

Thus, the model requires **4999 free parameters**.

### Part (c)

In [50]:
N = len(y_train)

pi_1 = np.sum(y_train == 1) / N
pi_0 = np.sum(y_train == 0) / N

print("Spam prior:", pi_1)
print("Non-spam prior:", pi_0)


Spam prior: 0.5
Non-spam prior: 0.5


In [51]:
##Estim
# ate conditional word probabilities using MLE

In [52]:
# Separate training emails by class
X_spam = X_train[y_train == 1]
X_nonspam = X_train[y_train == 0]

# Count occurrences of each vocabulary word
T_spam = np.sum(X_spam, axis=0)
T_nonspam = np.sum(X_nonspam, axis=0)

# Total word tokens in each class
total_spam = np.sum(T_spam)
total_nonspam = np.sum(T_nonspam)

# MLE conditional word probabilities
theta_spam = T_spam / total_spam
theta_nonspam = T_nonspam / total_nonspam

print("Spam probabilities shape:", theta_spam.shape)
print("Non-spam probabilities shape:", theta_nonspam.shape)
print("Spam probability sum:", theta_spam.sum())
print("Non-spam probability sum:", theta_nonspam.sum())


Spam probabilities shape: (2500,)
Non-spam probabilities shape: (2500,)
Spam probability sum: 1.0
Non-spam probability sum: 1.0


In [53]:
print("First 10 spam word counts:", T_spam[:10])
print("First 10 non-spam word counts:", T_nonspam[:10])

First 10 spam word counts: [1380.  954. 1159.   16. 1053.  923.  965.   33.  800.  828.]
First 10 non-spam word counts: [ 321.  252.  122. 1130.   58.  141.   70.  906.  186.  115.]


In [54]:
#print(len(T_spam))
#T_spam

In [55]:
#theta_spam

In [56]:
def compute_log_score(email,theta,prior):
    if np.any((email>0) & (theta ==0)):
        return -np.inf

    present = email >0

    score = np.log(prior) + np.sum(email[present] * np.log(theta[present]))

    return score


In [57]:
# test one
email = X_val[0]

score_spam = compute_log_score(email, theta_spam, pi_1)
score_nonspam = compute_log_score(email, theta_nonspam, pi_0)

print("Spam log score:", score_spam)
print("Non-spam log score:", score_nonspam)

Spam log score: -inf
Non-spam log score: -inf


In [58]:
def predict(email, theta_spam, theta_nonspam, pi_1, pi_0):

    score_spam = compute_log_score(email, theta_spam, pi_1)
    score_nonspam = compute_log_score(email, theta_nonspam, pi_0)

    if score_spam > score_nonspam:
        return 1
    else:
        return 0

In [59]:
y_pred_mle = np.array([
    predict(email, theta_spam, theta_nonspam, pi_1, pi_0)
    for email in X_val
])

print("Predictions shape:", y_pred_mle.shape)
print("First 10 predictions:", y_pred_mle[:10])

Predictions shape: (260,)
First 10 predictions: [0 0 0 0 0 0 0 0 0 0]


In [60]:
n_correct = np.sum(y_pred_mle == y_val)
n_incorrect = np.sum(y_pred_mle != y_val)

accuracy_mle = n_correct / len(y_val)

print("Correct predictions:", n_correct)
print("Incorrect predictions:", n_incorrect)
print(f"MLE Validation Accuracy: {accuracy_mle * 100:.2f}%")

Correct predictions: 130
Incorrect predictions: 130
MLE Validation Accuracy: 50.00%


In [61]:
both_inf = sum(
    np.isneginf(compute_log_score(email, theta_spam, pi_1))
    and np.isneginf(compute_log_score(email, theta_nonspam, pi_0))
    for email in X_val
)

print("Emails with both scores -inf:", both_inf)

Emails with both scores -inf: 260



**Part (c) — MLE Multinomial Naive Bayes**

The Multinomial Naive Bayes classifier was trained using maximum likelihood estimation on the training dataset. Class priors were estimated from the class frequencies, and class-conditional word probabilities were estimated from word occurrence counts.

On the 260 validation emails, the classifier achieved an accuracy of **50.00%**, with **130 correct predictions and 130 incorrect predictions**.

All 260 validation emails received log scores of negative infinity for both classes due to zero estimated word probabilities. Following the specified tie-breaking rule, all emails were classified as non-spam (label 0).

This demonstrates the zero-frequency problem of unsmoothed MLE estimation, which can make the classifier unable to distinguish between classes.

### Part (d) — MAP Estimation with Dirichlet Prior

In [62]:
alpha = 1
V = X_train.shape[1]

theta_spam_map = (T_spam + alpha) / (total_spam + alpha * V)
theta_nonspam_map = (T_nonspam + alpha) / (total_nonspam + alpha * V)

print("First 10 MAP spam probabilities:", theta_spam_map[:10])
print("First 10 MAP non-spam probabilities:", theta_nonspam_map[:10])

print("Spam probability sum:", theta_spam_map.sum())
print("Non-spam probability sum:", theta_nonspam_map.sum())

print("Zero probabilities in Spam:", np.sum(theta_spam_map == 0))
print("Zero probabilities in Non-spam:", np.sum(theta_nonspam_map == 0))

First 10 MAP spam probabilities: [0.01468118 0.01015245 0.01233177 0.00018072 0.0112049  0.00982289
 0.01026939 0.00036145 0.0085153  0.00881296]
First 10 MAP non-spam probabilities: [0.00501152 0.00393762 0.00191434 0.01760256 0.00091826 0.00221005
 0.00110502 0.01411629 0.00291042 0.00180539]
Spam probability sum: 1.0
Non-spam probability sum: 1.0
Zero probabilities in Spam: 0
Zero probabilities in Non-spam: 0


In [63]:
# Predict validation labels using MAP estimates
y_pred_map = np.array([
    predict(email, theta_spam_map, theta_nonspam_map, pi_1, pi_0)
    for email in X_val
])

print("MAP predictions shape:", y_pred_map.shape)
print("First 10 MAP predictions:", y_pred_map[:10])

MAP predictions shape: (260,)
First 10 MAP predictions: [0 0 0 0 0 0 0 0 0 0]


In [64]:
# Evaluate MAP classifier
n_correct_map = np.sum(y_pred_map == y_val)
n_incorrect_map = np.sum(y_pred_map != y_val)

accuracy_map = n_correct_map / len(y_val)

print("Correct predictions:", n_correct_map)
print("Incorrect predictions:", n_incorrect_map)
print(f"MAP Validation Accuracy: {accuracy_map * 100:.2f}%")

Correct predictions: 253
Incorrect predictions: 7
MAP Validation Accuracy: 97.31%


In [65]:
print(f"MLE Accuracy: {accuracy_mle * 100:.2f}%")
print(f"MAP Accuracy: {accuracy_map * 100:.2f}%")

improvement = (accuracy_map - accuracy_mle) * 100

print(f"Accuracy improvement: {improvement:.2f} percentage points")

MLE Accuracy: 50.00%
MAP Accuracy: 97.31%
Accuracy improvement: 47.31 percentage points


Using a symmetric Dirichlet(2, ..., 2) prior, corresponding to additive smoothing with α = 1, the MAP classifier achieved **97.31% validation accuracy**, correctly classifying 253 out of 260 emails, with only 7 incorrect predictions.

In comparison, the MLE classifier achieved 50.00% accuracy. MAP improved validation accuracy by 47.31 percentage points.

This improvement occurs because additive smoothing prevents zero class-conditional word probabilities. Unlike the MLE classifier, which produced negative-infinity scores for both classes on every validation email, the MAP classifier can calculate finite log scores and distinguish between spam and non-spam emails.

### Part (e) — Rank the five most discriminative features

In [66]:
log_ratio = np.log(theta_spam_map/theta_nonspam_map)
print("Log-ratio shape:", log_ratio.shape)
print("First 10 log-ratios:", log_ratio[:10])

Log-ratio shape: (2500,)
First 10 log-ratios: [ 1.0748278   0.94713804  1.86280712 -4.57882794  2.50162647  1.4917012
  2.22930015 -3.66496574  1.07356852  1.58544615]


In [67]:
abs_log_ratio = np.abs(log_ratio)

top5_indices = np.argsort(abs_log_ratio)[::-1][:5]

print("Top 5 vocabulary indices:", top5_indices + 1)
print("Top 5 signed log-ratios:", log_ratio[top5_indices])

Top 5 vocabulary indices: [ 29  47  88 123 210]
Top 5 signed log-ratios: [-6.55079454 -6.27006177 -5.91851808 -5.69438979 -5.43742962]


In [68]:
top5_table = pd.DataFrame({
    "Vocabulary Index": top5_indices + 1,
    "Signed Log-Ratio": log_ratio[top5_indices],
    "Favored Class": np.where(
        log_ratio[top5_indices] > 0,
        "Spam",
        np.where(log_ratio[top5_indices] < 0, "Non-spam", "Neither")
    )
})

print(top5_table.to_string(index=False))

 Vocabulary Index  Signed Log-Ratio Favored Class
               29         -6.550795      Non-spam
               47         -6.270062      Non-spam
               88         -5.918518      Non-spam
              123         -5.694390      Non-spam
              210         -5.437430      Non-spam


**Part (e) — Feature Ranking**

Using the MAP-smoothed class-conditional word probabilities from Part (d), I calculated the signed log-ratio for each of the 2,500 vocabulary features:

$$
r_j=\log\left(\frac{\hat\theta_{j|y=1}^{MAP}}{\hat\theta_{j|y=0}^{MAP}}\right).
$$

The features were ranked by the absolute values of their log-ratios. The five highest-ranked features are:

| Vocabulary Index | Signed Log-Ratio | Favored Class |
|---|---|---|
| 29 | -6.550795 | Non-spam |
| 47 | -6.270062 | Non-spam |
| 88 | -5.918518 | Non-spam |
| 123 | -5.694390 | Non-spam |
| 210 | -5.437430 | Non-spam |

All five features have negative log-ratios, indicating that they are more strongly associated with the non-spam class. Feature 29 has the largest absolute log-ratio and therefore provides the strongest individual class preference among the vocabulary features under this ranking criterion.